# E6 | Model Forecast Prophet
Treinar Prophet para D+1 e D+7 com MLflow

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error
import mlflow

load_dotenv()
print('✅ Imports OK')

In [ ]:
# Conectar RDS
RDS_HOST = os.getenv('RDS_HOST')
RDS_USER = os.getenv('RDS_USER')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE')

conn_str = f'postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:5432/{RDS_DATABASE}'
engine = create_engine(conn_str)
print('✅ RDS Connected')

In [ ]:
# Ler dados
df = pd.read_sql('SELECT * FROM gold.ml_forecast_dataset ORDER BY data_abertura', engine)
print(f'Loaded {len(df)} days from {df.data_abertura.min()} to {df.data_abertura.max()}')

In [ ]:
# Preparar Prophet dataset
df_p = df[['data_abertura', 'total_chamados']].rename(columns={'data_abertura': 'ds', 'total_chamados': 'y'})
df_p['ds'] = pd.to_datetime(df_p['ds'])
df_p['fim_de_semana'] = df['is_fim_de_semana'].values
df_p['taxa_violacao'] = df['pct_violacao_sla'].fillna(df['pct_violacao_sla'].mean()).values
df_p = df_p.sort_values('ds').reset_index(drop=True)
print(f'Ready: {df_p.shape}')

In [ ]:
# Split
train = df_p.iloc[:-30]
test = df_p.iloc[-30:]
print(f'Train: {len(train)}, Test: {len(test)}')

In [ ]:
# Treinar
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, seasonality_mode='additive')
model.add_regressor('fim_de_semana')
model.add_regressor('taxa_violacao')
print('Training...')
model.fit(train)
print('✅ Done')

In [ ]:
# Avaliar
forecast = model.predict(test[['ds', 'fim_de_semana', 'taxa_violacao']])
eval = test[['y']].copy()
eval['yhat'] = forecast['yhat'].values

mape = mean_absolute_percentage_error(eval['y'], eval['yhat'])
mae = (eval['y'] - eval['yhat']).abs().mean()
print(f'MAPE: {mape:.2%}, MAE: {mae:.0f}')

In [ ]:
# Forecast D+1 a D+7
future_dates = pd.date_range(start=pd.Timestamp(datetime.now()).normalize() + timedelta(days=1), periods=7)
future = pd.DataFrame({
    'ds': future_dates,
    'fim_de_semana': [1 if d.dayofweek in [5,6] else 0 for d in future_dates],
    'taxa_violacao': df_p['taxa_violacao'].mean()
})

forecast_future = model.predict(future)
for i, row in forecast_future.iterrows():
    yhat = int(row['yhat'])
    date = row['ds'].strftime('%d/%m')
    print(f'D+{i+1} ({date}): {yhat} chamados')

In [ ]:
# MLflow
mlflow.set_experiment('prophet_forecast')
with mlflow.start_run():
    mlflow.log_params({'seasonality': 'additive', 'train_days': len(train)})
    mlflow.log_metrics({'mape': mape, 'mae': mae})
    mlflow.sklearn.log_model(model, 'model')
    print('✅ Logged')